# PatchCore + WideResNet50-2 Multilayer, x224

This notebook is the curated review notebook for the saved `x224` WRN50-2 Multilayer PatchCore sweep.

Default behavior (`RETRAIN = False`):
- load saved sweep CSVs and scores
- recompute confusion matrices and defect analysis
- save regenerated figures

Set `RETRAIN = True` to rebuild the memory bank from scratch.


## Imports and Paths

This cell loads the shared evaluation helpers and resolves the local artifact locations used by the review notebook.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.evaluation import summarize_threshold_metrics

ARTIFACT_ROOT = REPO_ROOT / "experiments/anomaly_detection/patchcore/wideresnet50/x224/multilayer/artifacts/patchcore-wideresnet50-multilayer"
RESULTS_DIR = ARTIFACT_ROOT / "results"
PLOTS_DIR = ARTIFACT_ROOT / "plots"
METADATA_PATH = REPO_ROOT / "data/processed/x224/wm811k/metadata_50k_5pct.csv"
SELECTED_VARIANT_NAME = None

# RETRAIN=False loads saved CSVs; RETRAIN=True rebuilds from scratch
RETRAIN = False
RENDER_ALL_CACHED_VARIANTS = True
VARIANTS_TO_RENDER: list[str] = []
VARIANT_COLOR_VAL = "#4d908e"
VARIANT_COLOR_NORMAL = "#577590"
VARIANT_COLOR_ANOMALY = "#e76f51"
VARIANT_COLOR_DEFECT = "#8ab17d"


In [ ]:
from __future__ import annotations

import json
import math
import pickle
import random
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.core.indexes as core_indexes
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from torchvision.models import Wide_ResNet50_2_Weights, wide_resnet50_2

LABEL_NORMAL = "none"
LABEL_DEFECT = "pattern"


from wafer_defect.data.wm811k import WaferMapDataset

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def resolve_device(device_name: str) -> torch.device:
    return torch.device("cuda" if device_name == "auto" and torch.cuda.is_available() else device_name)


def summarize_threshold_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, float | int | list[list[int]]]:
    predicted = (scores >= threshold).astype(int)
    precision = float(precision_score(labels, predicted, zero_division=0)); recall = float(recall_score(labels, predicted, zero_division=0)); f1 = float(0.0 if precision + recall == 0 else 2.0 * precision * recall / (precision + recall))
    return {"threshold": float(threshold), "precision": precision, "recall": recall, "f1": f1, "auroc": float(roc_auc_score(labels, scores)), "auprc": float(average_precision_score(labels, scores)), "predicted_anomalies": int(predicted.sum()), "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist()}

def sweep_threshold_metrics(labels: np.ndarray, scores: np.ndarray) -> tuple[pd.DataFrame, dict[str, float]]:
    rows = []
    for threshold in np.unique(scores):
        metrics = summarize_threshold_metrics(labels, scores, float(threshold))
        rows.append({"threshold": float(threshold), "precision": metrics["precision"], "recall": metrics["recall"], "f1": metrics["f1"], "predicted_anomalies": metrics["predicted_anomalies"]})
    sweep_df = pd.DataFrame(rows).sort_values(["f1", "precision", "recall"], ascending=False).reset_index(drop=True)
    return pd.DataFrame(rows), sweep_df.iloc[0].to_dict()

def _teacher_feature_dims(layer_names: list[str]) -> dict[str, int]: return {"layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048} | {}
def _teacher_spatial_size(input_size: int, layer_names: list[str]) -> int:
    downsample = {"layer1": 4, "layer2": 8, "layer3": 16, "layer4": 32}
    return max(1, input_size // min(downsample[name] for name in layer_names))

class WideResNet50_2MultiLayerExtractor(nn.Module):
    def __init__(self, teacher_layers: list[str], pretrained: bool = True, input_size: int = 224, freeze_backbone: bool = True, normalize_imagenet: bool = True) -> None:
        super().__init__(); self.teacher_layers = [str(x).lower() for x in teacher_layers]
        weights = Wide_ResNet50_2_Weights.DEFAULT if pretrained else None; backbone = wide_resnet50_2(weights=weights)
        original_conv = backbone.conv1; adapted_conv = nn.Conv2d(1, original_conv.out_channels, kernel_size=original_conv.kernel_size, stride=original_conv.stride, padding=original_conv.padding, bias=False)
        with torch.no_grad(): adapted_conv.weight.copy_(original_conv.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = adapted_conv; self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool); self.layer1 = backbone.layer1; self.layer2 = backbone.layer2; self.layer3 = backbone.layer3; self.layer4 = backbone.layer4
        self.input_size = int(input_size); self.normalize_imagenet = bool(normalize_imagenet); self.register_buffer("image_mean", torch.tensor([0.4490], dtype=torch.float32).view(1, 1, 1, 1)); self.register_buffer("image_std", torch.tensor([0.2260], dtype=torch.float32).view(1, 1, 1, 1))
        dim_map = {"layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048}; self.feature_dims = {name: dim_map[name] for name in self.teacher_layers}; self.output_spatial = _teacher_spatial_size(self.input_size, self.teacher_layers)
        if freeze_backbone:
            for parameter in self.parameters(): parameter.requires_grad = False
    def preprocess(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.input_size or x.shape[-2] != self.input_size: x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        return (x - self.image_mean) / self.image_std if self.normalize_imagenet else x
    def forward_feature_maps(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        outputs = {}; x = self.preprocess(x); x = self.stem(x); x = self.layer1(x); 
        if "layer1" in self.teacher_layers: outputs["layer1"] = x
        x = self.layer2(x); 
        if "layer2" in self.teacher_layers: outputs["layer2"] = x
        x = self.layer3(x); 
        if "layer3" in self.teacher_layers: outputs["layer3"] = x
        x = self.layer4(x); 
        if "layer4" in self.teacher_layers: outputs["layer4"] = x
        return outputs

class MultiLayerPatchCoreModel(nn.Module):
    def __init__(self, image_size: int, teacher_layers: list[str], memory_bank_size: int, reduction: str, topk_ratio: float, pretrained: bool = True, freeze_backbone: bool = True, backbone_input_size: int = 224, normalize_imagenet: bool = True, query_chunk_size: int = 1024, memory_chunk_size: int = 4096) -> None:
        super().__init__(); self.teacher_layers = [str(x).lower() for x in teacher_layers]; self.memory_bank_size = int(memory_bank_size); self.reduction = str(reduction); self.topk_ratio = float(topk_ratio); self.query_chunk_size = int(query_chunk_size); self.memory_chunk_size = int(memory_chunk_size)
        self.teacher = WideResNet50_2MultiLayerExtractor(self.teacher_layers, pretrained=pretrained, input_size=backbone_input_size, freeze_backbone=freeze_backbone, normalize_imagenet=normalize_imagenet)
        self.feature_dim = sum(self.teacher.feature_dims.values()); self.reduced_spatial = self.teacher.output_spatial; self.register_buffer("memory_bank", torch.empty(0, self.feature_dim))
    @property
    def patches_per_image(self) -> int: return self.reduced_spatial * self.reduced_spatial
    def patch_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        feature_maps = self.teacher.forward_feature_maps(x); target_size = max(feature_map.shape[-1] for feature_map in feature_maps.values()); embeddings = []
        for layer_name in self.teacher_layers:
            feature_map = feature_maps[layer_name]
            if feature_map.shape[-1] != target_size or feature_map.shape[-2] != target_size: feature_map = F.interpolate(feature_map, size=(target_size, target_size), mode="bilinear", align_corners=False)
            layer_embeddings = feature_map.permute(0, 2, 3, 1).reshape(x.shape[0], -1, feature_map.shape[1]); embeddings.append(F.normalize(layer_embeddings, p=2, dim=-1))
        return F.normalize(torch.cat(embeddings, dim=-1), p=2, dim=-1)
    def set_memory_bank(self, memory_bank: torch.Tensor) -> None: self.memory_bank = F.normalize(memory_bank.to(dtype=torch.float32), p=2, dim=1).to(device=self.memory_bank.device, dtype=self.memory_bank.dtype)
    def nearest_patch_distances(self, patch_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size, patch_count, _ = patch_embeddings.shape; flat_queries = patch_embeddings.reshape(-1, self.feature_dim); all_mins = []
        for query_start in range(0, flat_queries.shape[0], self.query_chunk_size):
            query_chunk = flat_queries[query_start: query_start + self.query_chunk_size]; chunk_best = None
            for memory_start in range(0, self.memory_bank.shape[0], self.memory_chunk_size):
                memory_chunk = self.memory_bank[memory_start: memory_start + self.memory_chunk_size]; distances = torch.cdist(query_chunk, memory_chunk); current_best = distances.min(dim=1).values; chunk_best = current_best if chunk_best is None else torch.minimum(chunk_best, current_best)
            all_mins.append(chunk_best)
        return torch.cat(all_mins, dim=0).reshape(batch_size, patch_count)
    def reduce_patch_distances(self, patch_distances: torch.Tensor) -> torch.Tensor:
        if self.reduction == "max": return patch_distances.max(dim=1).values
        if self.reduction == "mean": return patch_distances.mean(dim=1)
        topk = max(1, int(math.ceil(patch_distances.shape[1] * self.topk_ratio))); return torch.topk(patch_distances, k=topk, dim=1).values.mean(dim=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.reduce_patch_distances(self.nearest_patch_distances(self.patch_embeddings(x)))

def sample_memory_indices(dataset_size: int, memory_bank_size: int, patches_per_image: int, seed: int) -> np.ndarray:
    image_count = min(dataset_size, max(1, math.ceil(memory_bank_size / patches_per_image))); rng = np.random.default_rng(seed); return np.sort(rng.choice(dataset_size, size=image_count, replace=False))
def build_memory_subset(dataset: Dataset, memory_bank_size: int, patches_per_image: int, seed: int) -> Subset: return Subset(dataset, sample_memory_indices(len(dataset), memory_bank_size, patches_per_image, seed).tolist())
def collect_memory_bank(model: MultiLayerPatchCoreModel, dataloader: DataLoader, device: torch.device, target_size: int, seed: int) -> torch.Tensor:
    patch_batches = []; model.eval()
    with torch.inference_mode():
        for inputs, labels in tqdm(dataloader, desc="Collect memory bank"):
            inputs = inputs.to(device); labels = labels.to(device); normal_mask = labels == 0
            if not torch.any(normal_mask): continue
            patch_batches.append(model.patch_embeddings(inputs[normal_mask]).reshape(-1, model.feature_dim).cpu())
    memory_bank = torch.cat(patch_batches, dim=0)
    if memory_bank.shape[0] > target_size:
        generator = torch.Generator().manual_seed(seed); keep = torch.randperm(memory_bank.shape[0], generator=generator)[:target_size]; memory_bank = memory_bank[keep]
    return memory_bank
def collect_scores(model: MultiLayerPatchCoreModel, dataloader: DataLoader, device: torch.device) -> pd.DataFrame:
    rows = []; model.eval()
    with torch.inference_mode():
        for inputs, labels in tqdm(dataloader, desc="Score batches"):
            scores = model(inputs.to(device))
            for score, label in zip(scores.cpu().tolist(), labels.tolist()): rows.append({"score": float(score), "is_anomaly": int(label)})
    return pd.DataFrame(rows)

def build_patchcore_checkpoint(model: MultiLayerPatchCoreModel, variant: dict[str, Any], threshold: float) -> dict[str, Any]:
    return {
        "name": str(variant["name"]),
        "teacher_backbone": "wideresnet50_2",
        "teacher_layers": list(TEACHER_LAYERS),
        "image_size": int(IMAGE_SIZE),
        "backbone_input_size": int(TEACHER_INPUT_SIZE),
        "pretrained": bool(PRETRAINED),
        "freeze_backbone": bool(FREEZE_BACKBONE),
        "normalize_imagenet": bool(NORMALIZE_IMAGENET),
        "memory_bank_size": int(variant["memory_bank_size"]),
        "reduction": str(variant["reduction"]),
        "topk_ratio": float(variant["topk_ratio"]),
        "query_chunk_size": int(QUERY_CHUNK_SIZE),
        "memory_chunk_size": int(MEMORY_CHUNK_SIZE),
        "feature_dim": int(model.feature_dim),
        "patches_per_image": int(model.patches_per_image),
        "threshold_quantile": float(THRESHOLD_QUANTILE),
        "threshold": float(threshold),
        "memory_bank": model.memory_bank.detach().cpu(),
    }

def save_patchcore_checkpoint(variant_output_dir: Path, model: MultiLayerPatchCoreModel, variant: dict[str, Any], threshold: float) -> Path:
    checkpoint_path = variant_output_dir / "patchcore_checkpoint.pt"
    torch.save(build_patchcore_checkpoint(model, variant, threshold), checkpoint_path)
    return checkpoint_path

In [ ]:
SWEEP_VARIANTS_X224 = [
    {"name": "topk_mb50k_r005_x224", "memory_bank_size": 600_000, "reduction": "topk_mean", "topk_ratio": 0.05},
    {"name": "topk_mb50k_r010_x224", "memory_bank_size": 600_000, "reduction": "topk_mean", "topk_ratio": 0.10},
]
IMAGE_SIZE = 224
BATCH_SIZE = 25
NUM_WORKERS = 0
TEACHER_LAYERS = ["layer2", "layer3"]
TEACHER_INPUT_SIZE = 224
PRETRAINED = True
FREEZE_BACKBONE = True
NORMALIZE_IMAGENET = True
QUERY_CHUNK_SIZE = 1024
MEMORY_CHUNK_SIZE = 4096
THRESHOLD_QUANTILE = 0.95
SEED = 42

sweep_results_csv = RESULTS_DIR / "patchcore_sweep_results.csv"

if not RETRAIN and sweep_results_csv.exists():
    print(f"Loaded saved sweep results from {sweep_results_csv}")
    print("Set RETRAIN = True to rebuild all variants from scratch.")
    sweep_results_df = pd.read_csv(sweep_results_csv)
    display(sweep_results_df)
else:
    set_seed(SEED)
    device = resolve_device("auto")
    print(f"Training on device: {device}")

    train_dataset   = WaferMapDataset(METADATA_PATH, split="train", image_size=IMAGE_SIZE)
    val_dataset     = WaferMapDataset(METADATA_PATH, split="val",   image_size=IMAGE_SIZE)
    test_dataset_tr = WaferMapDataset(METADATA_PATH, split="test",  image_size=IMAGE_SIZE)
    train_loader = DataLoader(train_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_dataset,     batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_dataset_tr, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print(f"train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset_tr)}")

    sweep_rows = []
    for variant in SWEEP_VARIANTS_X224:
        vname = variant["name"]
        print(f"\n=== PatchCore variant: {vname} ===")
        model = MultiLayerPatchCoreModel(
            image_size=IMAGE_SIZE, teacher_layers=TEACHER_LAYERS,
            memory_bank_size=variant["memory_bank_size"], reduction=variant["reduction"],
            topk_ratio=variant["topk_ratio"], pretrained=PRETRAINED,
            freeze_backbone=FREEZE_BACKBONE, backbone_input_size=TEACHER_INPUT_SIZE,
            normalize_imagenet=NORMALIZE_IMAGENET, query_chunk_size=QUERY_CHUNK_SIZE,
            memory_chunk_size=MEMORY_CHUNK_SIZE,
        ).to(device)
        memory_subset = build_memory_subset(train_dataset, variant["memory_bank_size"], model.patches_per_image, SEED)
        memory_loader = DataLoader(memory_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        model.set_memory_bank(collect_memory_bank(model, memory_loader, device, target_size=variant["memory_bank_size"], seed=SEED))
        val_scores_df  = collect_scores(model, val_loader,  device)
        test_scores_df = collect_scores(model, test_loader, device)
        threshold = float(val_scores_df.loc[val_scores_df["is_anomaly"] == 0, "score"].quantile(THRESHOLD_QUANTILE))
        labels  = test_scores_df["is_anomaly"].to_numpy()
        scores  = test_scores_df["score"].to_numpy()
        metrics = summarize_threshold_metrics(labels, scores, threshold)
        threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)

        # Save in the layout expected by load_variant_outputs()
        eval_dir = ARTIFACT_ROOT / vname / "results" / "evaluation"
        eval_dir.mkdir(parents=True, exist_ok=True)
        val_scores_df.to_csv(eval_dir / "val_scores.csv",        index=False)
        test_scores_df.to_csv(eval_dir / "test_scores.csv",      index=False)
        threshold_sweep_df.to_csv(eval_dir / "threshold_sweep.csv", index=False)
        summary = {
            "name": vname, "teacher_backbone": "wideresnet50_2", "teacher_layers": TEACHER_LAYERS,
            "memory_bank_size": int(variant["memory_bank_size"]),
            "memory_subset_images": int(len(memory_subset)),
            "patches_per_image": int(model.patches_per_image),
            "feature_dim": int(model.feature_dim),
            "reduction": variant["reduction"], "topk_ratio": float(variant["topk_ratio"]),
            "threshold_quantile": THRESHOLD_QUANTILE, "threshold": float(threshold),
            "metrics_at_validation_threshold": metrics,
            "best_threshold_sweep": {
                "threshold": float(best_sweep["threshold"]),
                "precision": float(best_sweep["precision"]),
                "recall":    float(best_sweep["recall"]),
                "f1":        float(best_sweep["f1"]),
                "predicted_anomalies": float(best_sweep["predicted_anomalies"]),
            },
        }
        (ARTIFACT_ROOT / vname / "results" / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        sweep_rows.append({
            "name": vname, "memory_bank_size": int(variant["memory_bank_size"]),
            "reduction": variant["reduction"], "topk_ratio": float(variant["topk_ratio"]),
            "threshold": float(threshold), "precision": float(metrics["precision"]),
            "recall": float(metrics["recall"]), "f1": float(metrics["f1"]),
            "auroc": float(metrics["auroc"]), "auprc": float(metrics["auprc"]),
            "best_sweep_f1": float(best_sweep["f1"]),
            "predicted_anomalies": int(metrics["predicted_anomalies"]),
        })
        print(sweep_rows[-1])

    sweep_results_df = pd.DataFrame(sweep_rows).sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    sweep_results_df.to_csv(sweep_results_csv, index=False)
    (RESULTS_DIR / "patchcore_sweep_summary.json").write_text(
        json.dumps({
            "sweep_variants": [v["name"] for v in SWEEP_VARIANTS_X224],
            "base_output_dir": str(ARTIFACT_ROOT),
            "teacher_backbone": "wideresnet50_2", "teacher_layers": TEACHER_LAYERS,
            "best_variant": sweep_results_df.iloc[0].to_dict(),
        }, indent=2),
        encoding="utf-8",
    )
    print(f"Sweep complete. Saved to {sweep_results_csv}")
    display(sweep_results_df)


## Metadata and Sweep Loading

This cell loads the saved sweep table, selects the best variant by default, and attaches the local `x224` benchmark metadata for downstream analysis.

In [ ]:
metadata = pd.read_csv(METADATA_PATH)
test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True)

sweep_results_df = pd.read_csv(RESULTS_DIR / "patchcore_sweep_results.csv")
sweep_summary = json.loads((RESULTS_DIR / "patchcore_sweep_summary.json").read_text(encoding="utf-8"))
selected_variant_name = str(SELECTED_VARIANT_NAME or sweep_summary["best_variant"]["name"])

display(metadata["split"].value_counts().rename_axis("split").to_frame("count"))
display(sweep_results_df)
print(f"Selected variant: {selected_variant_name}")


## Variant Loaders

These helpers normalize the per-variant artifact layout, recompute threshold metrics from the saved score CSVs, and render plots back into the variant folders.

In [ ]:
def load_variant_outputs(variant_name: str) -> dict[str, object]:
    variant_root = ARTIFACT_ROOT / variant_name
    summary_path = variant_root / "results" / "summary.json"
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing summary for {variant_name}: {summary_path}")

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    val_scores_df = pd.read_csv(variant_root / "results" / "evaluation" / "val_scores.csv")
    test_scores_df = pd.read_csv(variant_root / "results" / "evaluation" / "test_scores.csv")
    threshold_sweep_df = pd.read_csv(variant_root / "results" / "evaluation" / "threshold_sweep.csv")

    threshold = float(summary["threshold"])
    metrics = summarize_threshold_metrics(
        test_scores_df["is_anomaly"].to_numpy(),
        test_scores_df["score"].to_numpy(),
        threshold,
    )
    best_sweep = threshold_sweep_df.sort_values("f1", ascending=False).iloc[0].to_dict()
    defect_breakdown_path = variant_root / "results" / "evaluation" / "selected_defect_breakdown.csv"
    defect_breakdown_df = pd.read_csv(defect_breakdown_path) if defect_breakdown_path.exists() else None
    return {
        "summary": summary,
        "val_scores_df": val_scores_df,
        "test_scores_df": test_scores_df,
        "threshold_sweep_df": threshold_sweep_df,
        "metrics": metrics,
        "best_sweep": best_sweep,
        "variant_root": variant_root,
        "defect_breakdown_df": defect_breakdown_df,
    }


def compute_failure_tables(test_metadata: pd.DataFrame, test_scores_df: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    analysis_df = test_metadata.copy()
    analysis_df["score"] = test_scores_df.reset_index(drop=True)["score"]
    analysis_df["predicted_anomaly"] = (analysis_df["score"] > threshold).astype(int)
    analysis_df["error_type"] = "tn"
    analysis_df.loc[(analysis_df["is_anomaly"] == 0) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "fp"
    analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 0), "error_type"] = "fn"
    analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "tp"

    error_summary_df = (
        analysis_df.groupby("error_type")
        .agg(count=("error_type", "size"), mean_score=("score", "mean"))
        .reindex(["tp", "fn", "fp", "tn"])
    )

    defect_recall_df = (
        analysis_df[analysis_df["is_anomaly"] == 1]
        .groupby("defect_type")
        .agg(count=("defect_type", "size"), detected=("predicted_anomaly", "sum"), mean_score=("score", "mean"))
        .sort_values(["detected", "count"], ascending=[False, False])
    )
    defect_recall_df["recall"] = defect_recall_df["detected"] / defect_recall_df["count"]
    return analysis_df, error_summary_df, defect_recall_df


def render_variant_artifacts(variant_name: str, payload: dict[str, object]) -> dict[str, str]:
    summary = payload["summary"]
    threshold = float(summary["threshold"])
    val_scores_df = payload["val_scores_df"]
    test_scores_df = payload["test_scores_df"]
    threshold_sweep_df = payload["threshold_sweep_df"]
    metrics = payload["metrics"]
    best_sweep = payload["best_sweep"]
    variant_root = payload["variant_root"]
    variant_plots_dir = variant_root / "plots"
    variant_eval_dir = variant_root / "results" / "evaluation"
    variant_plots_dir.mkdir(exist_ok=True)
    variant_eval_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_scores_df["score"], bins=30, alpha=0.85, color=VARIANT_COLOR_VAL)
    axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
    axes[0].set_title(f"Validation Normal Score Distribution\n{variant_name}")
    axes[0].legend()

    axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal", color=VARIANT_COLOR_NORMAL)
    axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly", color=VARIANT_COLOR_ANOMALY)
    axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
    axes[1].set_title(f"Test Score Distribution\n{variant_name}")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / "score_distribution.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
    ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
    ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
    ax.axvline(threshold, color="red", linestyle="--", label=f"validation threshold = {threshold:.4f}")
    ax.axvline(best_sweep["threshold"], color="green", linestyle=":", label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
    ax.set_title(f"Threshold Sweep on Test Split\n{variant_name}")
    ax.set_xlabel("Anomaly-score threshold")
    ax.set_ylabel("Metric value")
    ax.legend()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / "threshold_sweep.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    cm_array = np.asarray(metrics["confusion_matrix"], dtype=float)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_array, cmap="Blues")
    ax.set_xticks([0, 1], labels=["pred_normal", "pred_anomaly"])
    ax.set_yticks([0, 1], labels=["true_normal", "true_anomaly"])
    ax.set_title(f"Confusion Matrix\n{variant_name}")
    for row_idx in range(cm_array.shape[0]):
        for col_idx in range(cm_array.shape[1]):
            ax.text(col_idx, row_idx, int(cm_array[row_idx, col_idx]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(variant_plots_dir / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_metadata, test_scores_df, threshold)
    analysis_df.to_csv(variant_eval_dir / "analysis_with_predictions.csv", index=False)
    error_summary_df.reset_index().to_csv(variant_eval_dir / "error_summary.csv", index=False)
    defect_recall_df.reset_index().to_csv(variant_eval_dir / "defect_recall.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(error_summary_df.index.astype(str), error_summary_df["count"], color=VARIANT_COLOR_ANOMALY)
    axes[0].set_title(f"Prediction Outcome Counts\n{variant_name}")
    axes[0].set_ylabel("count")
    top_defects_df = defect_recall_df.head(10).reset_index()
    axes[1].barh(top_defects_df["defect_type"], top_defects_df["recall"], color=VARIANT_COLOR_DEFECT)
    axes[1].set_xlim(0.0, 1.0)
    axes[1].set_title("Top Defect-Type Recall")
    axes[1].set_xlabel("recall")
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / "defect_breakdown.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    return {"plots_dir": str(variant_plots_dir), "evaluation_dir": str(variant_eval_dir)}


## Selected Variant Review

This cell loads the selected variant, shows the key metrics, and saves the main review plots into the branch-level `plots/` folder.

In [ ]:
selected_variant = load_variant_outputs(selected_variant_name)
summary = selected_variant["summary"]
val_scores_df = selected_variant["val_scores_df"]
test_scores_df = selected_variant["test_scores_df"]
threshold_sweep_df = selected_variant["threshold_sweep_df"]
metrics = selected_variant["metrics"]
best_sweep = selected_variant["best_sweep"]
threshold = float(summary["threshold"])

metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "auroc", "value": metrics["auroc"]},
        {"metric": "auprc", "value": metrics["auprc"]},
        {"metric": "threshold", "value": threshold},
    ]
)
confusion_df = pd.DataFrame(
    metrics["confusion_matrix"],
    index=["true_normal", "true_anomaly"],
    columns=["pred_normal", "pred_anomaly"],
)
display(metrics_df)
display(confusion_df)

plot_df = sweep_results_df.copy().sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
plot_df["label"] = plot_df["name"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(plot_df["label"], plot_df["f1"], color="#264653")
axes[0].set_title("WRN50-2 Multilayer PatchCore: F1")
axes[0].invert_yaxis()
axes[1].barh(plot_df["label"], plot_df["auroc"], color="#2a9d8f")
axes[1].set_title("WRN50-2 Multilayer PatchCore: AUROC")
axes[1].invert_yaxis()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "variant_comparison_metrics.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.85, color=VARIANT_COLOR_VAL)
axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[0].set_title(f"Validation Normal Score Distribution\n{selected_variant_name}")
axes[0].legend()
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal", color=VARIANT_COLOR_NORMAL)
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly", color=VARIANT_COLOR_ANOMALY)
axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
axes[1].set_title(f"Test Score Distribution\n{selected_variant_name}")
axes[1].legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "score_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
ax.axvline(threshold, color="red", linestyle="--", label=f"validation threshold = {threshold:.4f}")
ax.axvline(best_sweep["threshold"], color="green", linestyle=":", label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
ax.set_title(f"Threshold Sweep on Test Split\n{selected_variant_name}")
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


## Failure Analysis and Cached Variant Rendering

This section computes the selected variant’s defect-level behavior and then regenerates the same outputs for every saved variant folder from cached CSVs.

In [ ]:
analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(
    test_metadata,
    test_scores_df,
    threshold,
)
analysis_df.to_csv(RESULTS_DIR / "selected_analysis_with_predictions.csv", index=False)
error_summary_df.reset_index().to_csv(RESULTS_DIR / "selected_error_summary.csv", index=False)
defect_recall_df.reset_index().to_csv(RESULTS_DIR / "selected_defect_recall.csv", index=False)

display(error_summary_df)
display(defect_recall_df)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(error_summary_df.index.astype(str), error_summary_df["count"], color=VARIANT_COLOR_ANOMALY)
axes[0].set_title(f"Prediction Outcome Counts\n{selected_variant_name}")
axes[0].set_ylabel("count")
top_defects_df = defect_recall_df.head(10).reset_index()
axes[1].barh(top_defects_df["defect_type"], top_defects_df["recall"], color=VARIANT_COLOR_DEFECT)
axes[1].set_xlim(0.0, 1.0)
axes[1].set_title("Top Defect-Type Recall")
axes[1].set_xlabel("recall")
axes[1].invert_yaxis()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "defect_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

variant_names = sweep_results_df["name"].astype(str).tolist() if RENDER_ALL_CACHED_VARIANTS else []
variant_names.extend([str(name) for name in VARIANTS_TO_RENDER])
variant_names.append(selected_variant_name)
ordered_variant_names = []
seen = set()
for name in variant_names:
    if name not in seen:
        ordered_variant_names.append(name)
        seen.add(name)

rendered_rows = []
for variant_name in ordered_variant_names:
    payload = load_variant_outputs(variant_name)
    render_info = render_variant_artifacts(variant_name, payload)
    rendered_rows.append(
        {
            "variant_name": variant_name,
            "plots_dir": render_info["plots_dir"],
            "evaluation_dir": render_info["evaluation_dir"],
        }
    )

rendered_variants_df = pd.DataFrame(rendered_rows)
display(rendered_variants_df)


## Saved Outputs

This cell prints the final artifact locations for the curated review notebook.

In [ ]:
saved_outputs = {
    "artifact_root": str(ARTIFACT_ROOT),
    "results_dir": str(RESULTS_DIR),
    "plots_dir": str(PLOTS_DIR),
    "selected_variant_name": selected_variant_name,
    "rendered_variants": rendered_variants_df["variant_name"].tolist(),
}
saved_outputs
